# Base de datos y SQL - DataCo Smart Supply Chain

## 1. Análisis de claves y estructura de los datos

## 2. Diseño del modelo de base de datos

## 3. Creación de las tablas

## 4. Carga de los datos

## 5. Consultas SQL

## 6. Preparación de datos para Power BI

## 1. Análisis de claves y estructura de los datos

Antes de diseñar la base de datos es necesario analizar la estructura del dataset y determinar qué columnas pueden utilizarse como identificadores de las distintas entidades.

El dataset contiene información relacionada con clientes, pedidos, líneas de pedido, productos, categorías y departamentos. Sin embargo, al tratarse originalmente de un dataset en formato tabular, esta información se encuentra combinada en una misma tabla y algunos identificadores aparecen en varias columnas con nombres diferentes.

En esta primera fase se analizará la unicidad de los principales identificadores, la existencia de valores nulos y las relaciones entre las distintas entidades. El objetivo es determinar las posibles claves primarias y foráneas y establecer una estructura relacional adecuada antes de crear las tablas de la base de datos.


In [2]:
# Cargamos el dataset limpio que utilizaremos como base para diseñar la estructura de la base de datos
import pandas as pd
# 1. Cargar el CSV que exportaste desde el otro notebook
ml_delay_predictions = pd.read_csv("ml_delay_predictions.csv")

df = pd.read_csv("../data/processed/DataCoSupplyChain_clean.csv")

print("Dimensiones del dataset:", df.shape)
df.head()

Dimensiones del dataset: (180519, 51)


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_state,order_status,product_card_id,product_category_id,product_image,product_name,product_price,product_status,shipping_date_dateorders,shipping_mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class


### 1.1. Identificación de claves

En primer lugar, se analizan los principales identificadores del dataset para conocer su número de valores únicos y determinar qué función pueden desempeñar dentro del modelo relacional.

Una clave primaria debe identificar de forma única cada registro de una entidad. Por ello, se comprueba la cardinalidad de los identificadores de clientes, pedidos, líneas de pedido, productos, categorías y departamentos.

In [3]:
# Comprobamos cuántos valores únicos tiene cada identificador para conocer su posible función como clave
columnas_id = [
    "customer_id",
    "order_customer_id",
    "order_id",
    "order_item_id",
    "product_card_id",
    "order_item_cardprod_id",
    "category_id",
    "product_category_id",
    "department_id"
]

for col in columnas_id:
    print(f"{col}: {df[col].nunique()} valores únicos de {len(df)} registros")

customer_id: 20652 valores únicos de 180519 registros
order_customer_id: 20652 valores únicos de 180519 registros
order_id: 65752 valores únicos de 180519 registros
order_item_id: 180519 valores únicos de 180519 registros
product_card_id: 118 valores únicos de 180519 registros
order_item_cardprod_id: 118 valores únicos de 180519 registros
category_id: 51 valores únicos de 180519 registros
product_category_id: 51 valores únicos de 180519 registros
department_id: 11 valores únicos de 180519 registros


Los resultados muestran que order_item_id presenta 180.519 valores únicos, coincidiendo con el número total de registros del dataset. Esto indica que cada fila representa una línea de pedido diferente y que este identificador puede utilizarse como clave primaria de la tabla de detalle de pedidos.

Por otro lado, order_id presenta 65.752 valores únicos, por lo que un mismo pedido puede aparecer en varias filas. Esto es coherente con una estructura en la que un pedido puede contener varios productos.

customer_id identifica 20.652 clientes diferentes, mientras que product_card_id, category_id y department_id identifican respectivamente 118 productos, 51 categorías y 11 departamentos.

Además, se observa que algunas columnas presentan el mismo número de valores únicos y parecen representar el mismo identificador con diferentes nombres. Estas equivalencias se comprobarán antes de definir el modelo definitivo.

In [4]:
#comprobamos si las parejas de columnas que parecen duplicadas son realmente equivalentes.
print(
    "customer_id y order_customer_id coinciden:",
    (df["customer_id"] == df["order_customer_id"]).all()
)

print(
    "product_card_id y order_item_cardprod_id coinciden:",
    (df["product_card_id"] == df["order_item_cardprod_id"]).all()
)

print(
    "category_id y product_category_id coinciden:",
    (df["category_id"] == df["product_category_id"]).all()
)

customer_id y order_customer_id coinciden: True
product_card_id y order_item_cardprod_id coinciden: True
category_id y product_category_id coinciden: True


Las comprobaciones muestran que las tres parejas de identificadores son completamente equivalentes en todos los registros. Por tanto, no es necesario mantener ambas columnas en el modelo de base de datos.

Se utilizará customer_id como identificador del cliente, product_card_id como identificador del producto y category_id como identificador de la categoría. Las columnas duplicadas se excluirán posteriormente durante la transformación de los datos para evitar redundancia.

### 1.2. Comprobación de valores nulos en las claves

Después de identificar los posibles identificadores, debemos comprobar que no existen valores nulos en las columnas que utilizaremos como claves. Una clave primaria no debería contener valores nulos, ya que debe permitir identificar de forma inequívoca cada registro.

In [5]:
#comprobamos si existen NULL en esas posibles claves:
columnas_clave = [
    "customer_id",
    "order_id",
    "order_item_id",
    "product_card_id",
    "category_id",
    "department_id"
]

print(df[columnas_clave].isnull().sum())

customer_id        0
order_id           0
order_item_id      0
product_card_id    0
category_id        0
department_id      0
dtype: int64


No se han encontrado valores nulos en ninguno de los identificadores analizados. Por tanto, estas columnas pueden utilizarse como claves dentro del modelo sin necesidad de realizar un tratamiento específico de valores ausentes.

### 1.3. Análisis de las relaciones entre entidades

Una vez identificadas las posibles claves, es necesario comprobar cómo se relacionan entre sí las diferentes entidades.

En primer lugar, se analiza la relación entre productos, categorías y departamentos. El objetivo es comprobar si un mismo producto puede estar asociado a varias categorías o departamentos.

In [6]:
#comprobación de relaciones:
#comprobamos cuántos productos pertenecen a cada categoría:
print(
    df.groupby("product_card_id")["category_id"]
      .nunique()
      .value_counts()
      .sort_index()
)

category_id
1    118
Name: count, dtype: int64


In [7]:
#comprobamos cuántos productos pertenecen a cada departamento:
print(
    df.groupby("product_card_id")["department_id"]
      .nunique()
      .value_counts()
      .sort_index()
)

department_id
1    118
Name: count, dtype: int64


In [8]:
# Comprobamos cuántas líneas de pedido tiene cada pedido
lineas_por_pedido = df.groupby("order_id")["order_item_id"].nunique()

print("Pedidos:", lineas_por_pedido.size)
print("Máximo de líneas por pedido:", lineas_por_pedido.max())
print("Pedidos con más de una línea:", (lineas_por_pedido > 1).sum())

Pedidos: 65752
Máximo de líneas por pedido: 5
Pedidos con más de una línea: 45902


El análisis de los identificadores permite establecer las principales relaciones entre las entidades del dataset. Se ha comprobado que customer_id, product_card_id y category_id son equivalentes a sus respectivas columnas duplicadas, por lo que se utilizará un único identificador para cada entidad.

Las principales claves analizadas no presentan valores nulos. Además, order_item_id identifica de forma única cada una de las 180.519 líneas del dataset, mientras que order_id identifica 65.752 pedidos y puede aparecer en varias filas. En concreto, 45.902 pedidos contienen más de una línea y el máximo observado es de cinco líneas por pedido.

Por otro lado, cada producto está asociado a una única categoría y a un único departamento. Estos resultados permiten definir las relaciones principales del modelo relacional y sirven como base para el diseño de las tablas de la base de datos.

### 1.4. Análisis del nivel de detalle de las variables

Una vez analizadas las relaciones entre las principales entidades, es necesario determinar el nivel de detalle al que pertenece cada variable. Este análisis permite comprobar si una variable mantiene un único valor dentro de un pedido o si puede variar entre sus diferentes líneas de pedido.

Esta comprobación es especialmente importante para el diseño del modelo relacional, ya que una variable que presenta diferentes valores dentro de un mismo pedido no debe almacenarse como un único atributo de la entidad `orders`.

En primer lugar, se analizarán las variables `benefit_per_order`, `order_profit_per_order` y `sales_per_customer`, que por su denominación podrían interpretarse inicialmente como métricas asociadas al pedido o al cliente. Se comprobará cuántos valores diferentes presenta cada una dentro de un mismo `order_id`.


In [9]:
# Comprobamos si las métricas a nivel de pedido se mantienen constantes
# dentro de cada order_id

columnas_pedido = [
    "benefit_per_order",
    "order_profit_per_order",
    "sales_per_customer"
]

for col in columnas_pedido:
    valores_por_pedido = df.groupby("order_id")[col].nunique()

    print(f"\n{col}")
    print("Pedidos con un único valor:", (valores_por_pedido == 1).sum())
    print("Pedidos con varios valores:", (valores_por_pedido > 1).sum())


benefit_per_order
Pedidos con un único valor: 19850
Pedidos con varios valores: 45902

order_profit_per_order
Pedidos con un único valor: 19850
Pedidos con varios valores: 45902

sales_per_customer
Pedidos con un único valor: 19850
Pedidos con varios valores: 45902


Los resultados muestran que las tres variables presentan varios valores dentro de 45.902 pedidos y un único valor en los 19.850 pedidos restantes.

Por tanto, a pesar de que sus nombres pueden sugerir que se trata de métricas asociadas al pedido completo, en este dataset no se mantienen constantes en todas las líneas de un mismo pedido. Por este motivo, se conservarán en el nivel de detalle de order_items y no se tratarán como atributos únicos de orders.

A continuación, se comprueba el comportamiento de las variables directamente relacionadas con cada línea de pedido: sales, order_item_total, order_item_discount y order_item_quantity.

El objetivo es verificar que cada order_item_id presenta un único valor para estas variables, confirmando que su nivel de detalle corresponde efectivamente a la línea de pedido.

In [10]:
# Comprobamos las métricas relacionadas con cada línea de pedido
columnas_linea = [
    "sales",
    "order_item_total",
    "order_item_discount",
    "order_item_quantity"
]

for col in columnas_linea:
    valores_por_linea = df.groupby("order_item_id")[col].nunique()

    print(f"\n{col}")
    print("Líneas con un único valor:", (valores_por_linea == 1).sum())
    print("Líneas con varios valores:", (valores_por_linea > 1).sum())


sales
Líneas con un único valor: 180519
Líneas con varios valores: 0

order_item_total
Líneas con un único valor: 180519
Líneas con varios valores: 0

order_item_discount
Líneas con un único valor: 180519
Líneas con varios valores: 0

order_item_quantity
Líneas con un único valor: 180519
Líneas con varios valores: 0


Las cuatro variables presentan un único valor para las 180.519 líneas de pedido y ningún caso con varios valores. Esto confirma que sales, order_item_total, order_item_discount y order_item_quantity pertenecen al nivel de detalle de order_items y pueden almacenarse directamente en esta tabla.

### 1.5. Conclusiones

El análisis de las claves y de la estructura del dataset ha permitido identificar los principales elementos necesarios para construir posteriormente un modelo relacional.

En primer lugar, se han identificado los principales identificadores de clientes, pedidos, líneas de pedido, productos, categorías y departamentos. También se ha comprobado que algunas columnas representan el mismo identificador con diferentes nombres, concretamente `customer_id` y `order_customer_id`, `product_card_id` y `order_item_cardprod_id`, y `category_id` y `product_category_id`. Por ello, se evitará mantener estos identificadores duplicados en el modelo final.

Las principales claves analizadas no presentan valores nulos. Además, `order_item_id` identifica de forma única las 180.519 líneas del dataset, mientras que `order_id` identifica 65.752 pedidos y puede aparecer en varias filas. Esto confirma que un pedido puede estar compuesto por varias líneas de pedido.

También se ha comprobado que cada producto está asociado a una única categoría y a un único departamento, permitiendo establecer las relaciones correspondientes entre estas entidades.

Por último, el análisis del nivel de detalle ha permitido determinar que `sales`, `order_item_total`, `order_item_discount` y `order_item_quantity` presentan un único valor por línea de pedido. En cambio, `benefit_per_order`, `order_profit_per_order` y `sales_per_customer` pueden presentar diferentes valores dentro de un mismo pedido, por lo que se conservarán en el nivel de detalle de las líneas de pedido.

A partir de todos estos análisis, se utilizarán como identificadores principales del modelo `customer_id` para los clientes, `order_id` para los pedidos, `order_item_id` para las líneas de pedido, `product_card_id` para los productos, `category_id` para las categorías y `department_id` para los departamentos. Estos identificadores se utilizarán posteriormente para definir las claves primarias (PK) y las claves foráneas (FK) del modelo relacional.


## 2. Diseño del modelo de base de datos

### 2.1. Revisión de las variables disponibles

Antes de definir las tablas que compondrán la base de datos, se revisan todas las variables disponibles en el dataset limpio. El objetivo es conocer la información disponible y determinar posteriormente qué columnas corresponden a cada una de las entidades identificadas.

La siguiente comprobación muestra el listado completo de variables disponibles:


In [11]:
# Mostramos todas las columnas disponibles en el dataset
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")


1. type
2. days_for_shipping_real
3. days_for_shipment_scheduled
4. benefit_per_order
5. sales_per_customer
6. delivery_status
7. late_delivery_risk
8. category_id
9. category_name
10. customer_city
11. customer_country
12. customer_email
13. customer_fname
14. customer_id
15. customer_lname
16. customer_password
17. customer_segment
18. customer_state
19. customer_street
20. customer_zipcode
21. department_id
22. department_name
23. latitude
24. longitude
25. market
26. order_city
27. order_country
28. order_customer_id
29. order_date_dateorders
30. order_id
31. order_item_cardprod_id
32. order_item_discount
33. order_item_discount_rate
34. order_item_id
35. order_item_product_price
36. order_item_profit_ratio
37. order_item_quantity
38. sales
39. order_item_total
40. order_profit_per_order
41. order_region
42. order_state
43. order_status
44. product_card_id
45. product_category_id
46. product_image
47. product_name
48. product_price
49. product_status
50. shipping_date_dateorders
51

### 2.2. Definición de las tablas y distribución de variables

A partir del análisis realizado en el apartado anterior, se diseña un modelo relacional que permita organizar la información del dataset en diferentes entidades, reduciendo la redundancia y facilitando posteriormente la realización de consultas SQL y el análisis en Power BI.

La estructura se basa en seis tablas principales: `customers`, `orders`, `order_items`, `products`, `categories` y `departments`.

La separación de estas entidades responde a la naturaleza de los datos. La información de los clientes se almacena de forma independiente de sus pedidos, mientras que los pedidos se separan de sus correspondientes líneas de pedido. Del mismo modo, los productos se mantienen en una tabla independiente y se relacionan con sus categorías y departamentos.

Esta estructura permite representar las relaciones existentes en los datos sin repetir innecesariamente la información descriptiva. Además, cada tabla contará con una clave primaria (PK) que permitirá identificar de forma única sus registros, mientras que las claves foráneas (FK) permitirán establecer las relaciones entre las diferentes entidades.


| Tabla         | Contenido                                        | PK                |
| ------------- | ------------------------------------------------ | ----------------- |
| `customers`   | Información de los clientes                      | `customer_id`     |
| `orders`      | Información general de los pedidos               | `order_id`        |
| `order_items` | Detalle de cada línea de pedido                  | `order_item_id`   |
| `products`    | Información de los productos                     | `product_card_id` |
| `categories`  | Categorías de productos                          | `category_id`     |
| `departments` | Departamentos a los que pertenecen los productos | `department_id`   |


A continuación, se detalla la distribución de las variables disponibles entre las diferentes tablas. La asignación se realiza teniendo en cuenta la entidad a la que pertenece cada variable y las relaciones identificadas durante el análisis de la estructura de los datos

### 2.2.1. customers

Aquí metemos las variables que identifican y describen al cliente. Su clave primaria es customer_id, que permite identificar de forma única a cada cliente.

Variable	
- customer_id	PK. Identificador del cliente
- customer_fname	
- customer_lname	
- customer_email	
- customer_segment	
- customer_city	
- customer_state	
- customer_country	
- customer_street	
- customer_zipcode

#### 2.2.2. Tabla orders

Esta tabla contiene la información general asociada a cada pedido. Su clave primaria es order_id, que identifica de forma única cada pedido, mientras que  customer_id  actúa como clave foránea y permite relacionar cada pedido con el cliente que lo realizó.

Las variables asignadas a esta tabla son:
Variable	
- order_id	PK. Identificador del pedido
- customer_id	FK. Identificador del cliente
- type	
- order_date_dateorders	
- shipping_date_dateorders	
- days_for_shipping_real	
- days_for_shipment_scheduled	
- delivery_status	
- late_delivery_risk	
- shipping_mode	
- market	
- order_city	
- order_state	
- order_country	
- order_region	
- order_status	
- latitude	
- longitude	

Estas variables describen características generales de la transacción, su entrega y su destino, por lo que se mantienen a nivel de pedido.

Las variables `latitude` y `longitude` se mantienen en esta tabla debido a que el dataset las proporciona como coordenadas de la ubicación de la tienda asociada a la operación. Al no disponer de un identificador específico de tienda, no se crea una entidad independiente para las tiendas.

Por otro lado, `type` se incorpora a `orders` porque el diccionario de datos lo define como el tipo de transacción realizada.


#### 2.2.3. Tabla `order_items`

Esta tabla contiene el detalle de cada línea de pedido. Cada registro representa una línea individual y se identifica de forma única mediante `order_item_id`, que actúa como clave primaria.

La tabla se relaciona con `orders` mediante `order_id` y con `products` mediante `product_card_id`, ambos utilizados como claves foráneas.

Las variables asignadas a esta tabla son:
Variable
- order_item_id	PK. Identificador de la línea de pedido
- order_id	FK. Identificador del pedido
- product_card_id	FK. Identificador del producto
- order_item_quantity	
- order_item_product_price	
- order_item_discount	
- order_item_discount_rate
- order_item_total	
- order_item_profit_ratio	
- benefit_per_order	
- order_profit_per_order	
- sales	
- sales_per_customer	

Las variables relacionadas con cantidades, precios, descuentos, ventas y beneficios se mantienen en esta tabla porque corresponden al nivel de detalle de la línea de pedido.

En el análisis realizado anteriormente se comprobó que `sales`, `order_item_total`, `order_item_discount` y `order_item_quantity` presentan un único valor para cada `order_item_id`. Asimismo, se determinó que `benefit_per_order`, `order_profit_per_order` y `sales_per_customer` pueden presentar diferentes valores dentro de un mismo `order_id`. Por este motivo, estas variables se conservan en el nivel de detalle de `order_items` y no se trasladan a la tabla `orders`.


#### 2.2.4. Tabla `products`

Esta tabla contiene la información descriptiva de los productos disponibles en el catálogo. Su clave primaria es `product_card_id`, que identifica de forma única cada producto.

La tabla se relaciona con `categories` mediante `category_id` y con `departments` mediante `department_id`, ambos utilizados como claves foráneas.

Las variables asignadas a esta tabla son:

Variable

* `product_card_id`		PK. Identificador del producto
* `product_name`		
* `product_price`		
* `product_image`		
* `product_status`		
* `category_id`		FK. Identificador de la categoría
* `department_id`	FK. Identificador del departamento

Los identificadores `order_item_cardprod_id` y `product_category_id` no se incorporan a esta tabla porque contienen la misma información que `product_card_id` y `category_id`, respectivamente, tal como se comprobó en el análisis de las claves.

Por otro lado, `product_description` no forma parte del modelo final porque esta variable no está disponible en el dataset limpio utilizado para la construcción de la base de datos.

El análisis realizado previamente mostró que cada producto está asociado a una única categoría y a un único departamento, por lo que estas relaciones se representan mediante las correspondientes claves foráneas.


#### 2.2.5. Tabla categories

Esta tabla contiene las diferentes categorías de productos del catálogo. Su clave primaria es `category_id`, que identifica de forma única cada categoría.

La tabla se relaciona con `products` mediante `category_id`, que actúa como clave foránea en la tabla `products`.

Las variables asignadas a esta tabla son:

Variable

* `category_id`		PK. Identificador de la categoría
* `category_name`	

El análisis realizado previamente mostró que cada producto está asociado a una única categoría, mientras que una misma categoría puede estar asociada a varios productos. Por este motivo, la relación entre `categories` y `products` se establecerá como una relación de uno a muchos (1:N).


#### 2.2.6. Tabla `departments`

Esta tabla contiene los diferentes departamentos o áreas comerciales del catálogo de productos. Su clave primaria es `department_id`, que identifica de forma única cada departamento.

La tabla se relaciona con `products` mediante `department_id`, que actúa como clave foránea en la tabla `products`.

Las variables asignadas a esta tabla son:

Variable

* `department_id`		PK. Identificador del departamento
* `department_name`

Los valores disponibles incluyen departamentos como Fitness, Footwear, Apparel, Golf, Outdoors, Fan Shop, Book Shop, Discs Shop, Technology, Pet Shop y Health and Beauty.

El análisis realizado previamente mostró que cada producto está asociado a un único departamento, mientras que un mismo departamento puede estar asociado a varios productos. Por este motivo, la relación entre `departments` y `products` se establecerá como una relación de uno a muchos (1:N)


### 2.2.7. Variables excluidas del modelo

Por tanto, las cuatro quedan fuera por dos motivos distintos:
Variable
- order_customer_id	Duplicado de customer_id
- order_item_cardprod_id	Duplicado de product_card_id
- product_category_id	Duplicado de category_id
- customer_password	Sin utilidad analítica y no necesaria para el modelo

### 2.3. Definición de las relaciones entre tablas

Las tablas se relacionan mediante claves primarias (PK) y claves foráneas (FK), siguiendo la estructura identificada durante el análisis de los datos.

Las relaciones del modelo son:

* `customers` **1:N** `orders` mediante `customer_id`.
* `orders` **1:N** `order_items` mediante `order_id`.
* `products` **1:N** `order_items` mediante `product_card_id`.
* `categories` **1:N** `products` mediante `category_id`.
* `departments` **1:N** `products` mediante `department_id`.

De esta forma, `order_items` representa el nivel de mayor detalle del modelo, ya que cada registro corresponde a una línea individual de pedido.

El modelo permite relacionar clientes, pedidos, productos, categorías y departamentos evitando duplicar información descriptiva.


### 2.4. Esquema del modelo relacional

El modelo relacional diseñado queda estructurado de la siguiente forma:

`customers` → `orders` → `order_items` ← `products` ← `categories`
　　　　　　　　　　　　　　　　　　　↑
　　　　　　　　　　　　　　　　 `departments`

Las relaciones se establecen mediante las claves primarias y foráneas definidas en el apartado anterior. Este esquema servirá como referencia para la creación de las tablas y sus relaciones en la base de datos.



### 3.1. Creación de la base de datos

Una vez definido el modelo relacional, se procede a crear la base de datos que almacenará las tablas del modelo.

En este proyecto se utilizará **SQLite**, ya que permite trabajar con una base de datos relacional local de forma sencilla y sin necesidad de configurar un servidor.

Primero se establece la conexión con la base de datos.

La base de datos se almacenará en la carpeta `data/processed`, junto al dataset limpio utilizado en el proyecto.



In [12]:
import sqlite3
conn = sqlite3.connect("../data/processed/dataco_supply_chain.db")

### 3.2. Creación de las tablas

A partir del modelo relacional definido, se crean las seis tablas que compondrán la base de datos. Para cada tabla se establece su clave primaria (PK) y, cuando corresponde, las claves foráneas (FK) que permiten relacionarla con el resto de entidades.

Las tablas se crean en un orden que permite establecer correctamente las relaciones entre las claves primarias y foráneas.

In [13]:
# Crear un cursor para ejecutar las instrucciones SQL
cursor = conn.cursor()

# Crear las tablas del modelo relacional
# IF NOT EXISTS evita errores si las tablas ya han sido creadas
cursor.executescript("""
    
-- Tabla de clientes
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    customer_fname TEXT,
    customer_lname TEXT,
    customer_email TEXT,
    customer_segment TEXT,
    customer_city TEXT,
    customer_state TEXT,
    customer_country TEXT,
    customer_street TEXT,
    customer_zipcode INTEGER
);

-- Tabla de categorías
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT
);

-- Tabla de departamentos
CREATE TABLE IF NOT EXISTS departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT
);

-- Tabla de productos
-- Las claves foráneas relacionan cada producto con su categoría y departamento
CREATE TABLE IF NOT EXISTS products (
    product_card_id INTEGER PRIMARY KEY,
    product_name TEXT,
    product_price REAL,
    product_image TEXT,
    product_status INTEGER,
    category_id INTEGER,
    department_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id),
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

-- Tabla de pedidos
-- Cada pedido se relaciona con el cliente que lo realizó
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    type TEXT,
    order_date_dateorders TEXT,
    shipping_date_dateorders TEXT,
    days_for_shipping_real INTEGER,
    days_for_shipment_scheduled INTEGER,
    delivery_status TEXT,
    late_delivery_risk INTEGER,
    shipping_mode TEXT,
    market TEXT,
    order_city TEXT,
    order_state TEXT,
    order_country TEXT,
    order_region TEXT,
    order_status TEXT,
    latitude REAL,
    longitude REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

-- Tabla de líneas de pedido
-- Cada línea se relaciona con un pedido y con un producto
CREATE TABLE IF NOT EXISTS order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_card_id INTEGER,
    order_item_quantity INTEGER,
    order_item_product_price REAL,
    order_item_discount REAL,
    order_m_profit_ratio REAL,
    benefit_per_order REAL,
    order_profit_per_order REAL,
    sales REAL,
    sales_per_customer REAL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_card_id) REFERENCES products(product_card_id)
);

-- Tabla de predicciones del modelo de Machine Learning
-- Cada predicción corresponde a una línea de pedido concreta
CREATE TABLE IF NOT EXISTS ml_delay_predictions (
    order_item_id INTEGER PRIMARY KEY,
    predicted_delay INTEGER,
    delay_probability REAL,
    actual_late_delivery_risk INTEGER,
    FOREIGN KEY (order_item_id) REFERENCES order_items(order_item_id)
);

""")

# Confirmar los cambios realizados en la base de datos
conn.commit()

print("Tablas creadas correctamente.")

Tablas creadas correctamente.


### 3.3. Comprobación de la estructura de las tablas

Una vez creadas las tablas, se comprueba que todas se encuentran correctamente registradas en la base de datos.

In [14]:
# Consultar las tablas creadas en la base de datos
tablas = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

tablas


,name
0,categories
1,customers
2,departments
3,ml_delay_predictions
4,order_items
5,orders
6,products


### 3.4. Comprobación de claves y relaciones

Se comprueba que las claves primarias (PK) y claves foráneas (FK) se han definido correctamente en las tablas creadas.

In [15]:
# Comprobar las claves primarias y foráneas de cada tabla
for tabla in tablas["name"]:
    print(f"\n--- {tabla} ---")
    
    estructura = pd.read_sql_query(
        f"PRAGMA table_info({tabla});",
        conn
    )
    
    claves_foraneas = pd.read_sql_query(
        f"PRAGMA foreign_key_list({tabla});",
        conn
    )
    
    print("Clave primaria:")
    display(estructura[estructura["pk"] == 1][["name", "type"]])
    
    print("Claves foráneas:")
    display(claves_foraneas[["from", "table", "to"]])


--- categories ---
Clave primaria:


,name,type
0,category_id,INTEGER


Claves foráneas:


,from,table,to



--- customers ---
Clave primaria:


...

,from,table,to
0,customer_id,customers,customer_id



--- products ---
Clave primaria:


,name,type
0,product_card_id,INTEGER


Claves foráneas:


,from,table,to
0,department_id,departments,department_id
1,category_id,categories,category_id


Los resultados confirman que las claves primarias y foráneas se han definido correctamente según el modelo relacional diseñado. Las tablas `customers`, `categories` y `departments` no contienen claves foráneas porque actúan como entidades independientes en el lado 1 de las relaciones. Por su parte, `orders`, `products` y `order_items` contienen las claves foráneas necesarias para establecer las relaciones con las tablas correspondientes.

De este modo, la estructura de la base de datos queda correctamente definida y preparada para la carga de los datos.


## 4 Carga de los datos

### 4.1 Preparación de los datos para la carga
A partir del dataset limpio, se seleccionan las variables correspondientes a cada tabla del modelo relacional. Las columnas redundantes o excluidas durante el diseño no se incorporan a la base de datos.

In [16]:
# Crear un DataFrame para cada tabla seleccionando únicamente
# las columnas definidas en el modelo relacional

customers_df = df[
    [
        "customer_id",
        "customer_fname",
        "customer_lname",
        "customer_email",
        "customer_segment",
        "customer_city",
        "customer_state",
        "customer_country",
        "customer_street",
        "customer_zipcode"
    ]
].drop_duplicates()

categories_df = df[
    [
        "category_id",
        "category_name"
    ]
].drop_duplicates()

departments_df = df[
    [
        "department_id",
        "department_name"
    ]
].drop_duplicates()

products_df = df[
    [
        "product_card_id",
        "product_name",
        "product_price",
        "product_image",
        "product_status",
        "category_id",
        "department_id"
    ]
].drop_duplicates()

orders_df = df[
    [
        "order_id",
        "customer_id",
        "type",
        "order_date_dateorders",
        "shipping_date_dateorders",
        "days_for_shipping_real",
        "days_for_shipment_scheduled",
        "delivery_status",
        "late_delivery_risk",
        "shipping_mode",
        "market",
        "order_city",
        "order_state",
        "order_country",
        "order_region",
        "order_status",
        "latitude",
        "longitude"
    ]
].drop_duplicates()

order_items_df = df[
    [
        "order_item_id",
        "order_id",
        "product_card_id",
        "order_item_quantity",
        "order_item_product_price",
        "order_item_discount",
        "order_item_discount_rate",
        "order_item_total",
        "order_item_profit_ratio",
        "benefit_per_order",
        "order_profit_per_order",
        "sales",
        "sales_per_customer"
    ]
].copy()

# DataFrame con las predicciones generadas por el modelo de Machine Learning.
# Contiene una predicción para cada línea de pedido.

ml_delay_predictions_df = ml_delay_predictions[
    [
        "order_item_id",
        "predicted_delay",
        "delay_probability",
        "actual_late_delivery_risk"
    ]
].copy()

print("DataFrames preparados correctamente.")


DataFrames preparados correctamente.


### 4.2. Comprobación de los datos preparados

Antes de cargar los datos en la base de datos, se comprueba el número de registros de cada DataFrame para verificar que la separación se ha realizado correctamente.

In [17]:
registros = {
    "customers": len(customers_df),
    "categories": len(categories_df),
    "departments": len(departments_df),
    "products": len(products_df),
    "orders": len(orders_df),
    "order_items": len(order_items_df),
    "ml_delay_predictions": len(ml_delay_predictions_df)
}

registros

{'customers': 20652,
 'categories': 51,
 'departments': 11,
 'products': 118,
 'orders': 65752,
 'order_items': 180519,
 'ml_delay_predictions': 180519}

Los resultados coinciden con las cantidades identificadas durante el análisis inicial: 20.652 clientes, 51 categorías, 11 departamentos, 118 productos, 65.752 pedidos y 180.519 líneas de pedido.

Esto confirma que los datos se han distribuido correctamente entre las tablas del modelo y que la información redundante se ha eliminado de las entidades correspondientes.


### 4.3. Carga de los datos en SQLite

Una vez comprobada la correcta distribución de los datos, se cargan los seis DataFrames en las tablas correspondientes de la base de datos SQLite.

In [18]:
# Vaciar las tablas antes de realizar una nueva carga
# Se mantiene la estructura, las PK y las FK de las tablas.
#
# Se vacían primero las tablas que dependen de otras para respetar
# las relaciones definidas en el modelo relacional.

cursor.execute("DELETE FROM ml_delay_predictions")
cursor.execute("DELETE FROM order_items")
cursor.execute("DELETE FROM orders")
cursor.execute("DELETE FROM products")
cursor.execute("DELETE FROM categories")
cursor.execute("DELETE FROM departments")
cursor.execute("DELETE FROM customers")

# Cargar cada DataFrame en su tabla correspondiente
customers_df.to_sql("customers", conn, if_exists="append", index=False)
categories_df.to_sql("categories", conn, if_exists="append", index=False)
departments_df.to_sql("departments", conn, if_exists="append", index=False)
products_df.to_sql("products", conn, if_exists="append", index=False)
orders_df.to_sql("orders", conn, if_exists="append", index=False)
order_items_df.to_sql("order_items", conn, if_exists="append", index=False)
ml_delay_predictions_df.to_sql(
    "ml_delay_predictions",
    conn,
    if_exists="append",
    index=False
)

# Confirmar la carga de los datos
conn.commit()

print("Datos cargados correctamente en la base de datos.")

Datos cargados correctamente en la base de datos.


### 4.4. Comprobación de la carga de los datos

Una vez cargados los datos, se comprueba el número de registros almacenados en cada tabla de la base de datos. Los resultados se comparan con los registros preparados previamente para verificar que la carga se ha realizado correctamente.


In [19]:
# Comprobar el número de registros cargados en cada tabla
for tabla in tablas["name"]:
    resultado = pd.read_sql_query(
        f"SELECT COUNT(*) AS registros FROM {tabla};",
        conn
    )
    
    print(f"{tabla}: {resultado.loc[0, 'registros']:,} registros")

categories: 51 registros
customers: 20,652 registros
departments: 11 registros
ml_delay_predictions: 180,519 registros
order_items: 180,519 registros
orders: 65,752 registros
products: 118 registros


Los resultados coinciden con el número de registros de los DataFrames preparados previamente, confirmando que los datos se han cargado correctamente en las seis tablas de la base de datos.

La base de datos contiene actualmente 20.652 clientes, 51 categorías, 11 departamentos, 118 productos, 65.752 pedidos y 180.519 líneas de pedido.

De esta forma, la información del dataset limpio queda almacenada en una estructura relacional y preparada para realizar consultas mediante SQL.


## 5. Consultas SQL

Una vez creada y cargada la base de datos, se realizan diferentes consultas SQL con el objetivo de extraer información y obtener indicadores relevantes a partir de las tablas del modelo relacional.

Las consultas permitirán trabajar con operaciones básicas de selección y filtrado, funciones de agregación, agrupaciones y relaciones entre tablas mediante `JOIN`. De esta forma, se aplicará SQL sobre la estructura creada y se obtendrán resultados orientados al análisis de los datos y a la toma de decisiones.


### 5.1. Consultas básicas

En primer lugar, se realizan consultas sencillas sobre las tablas de la base de datos para comprobar su contenido y aplicar operaciones básicas de selección, filtrado y ordenación de registros.


In [20]:
# Consultar los primeros pedidos de la base de datos
consulta = """
SELECT 
    order_id,
    customer_id,
    order_date_dateorders,
    order_status,
    market
FROM orders
LIMIT 10;
"""

resultado = pd.read_sql_query(consulta, conn)

resultado

,order_id,customer_id,order_date_dateorders,order_status,market
0,1,11599,2015-01-01 00:00:00,CLOSED,LATAM
1,2,256,2015-01-01 00:21:00,PENDING_PAYMENT,LATAM
2,4,8827,2015-01-01 01:03:00,CLOSED,LATAM
3,5,11318,2015-01-01 01:24:00,COMPLETE,LATAM
4,7,4530,2015-01-01 02:06:00,COMPLETE,LATAM
5,8,2911,2015-01-01 02:27:00,PROCESSING,LATAM
6,9,5657,2015-01-01 02:48:00,PENDING_PAYMENT,LATAM
7,10,5648,2015-01-01 03:09:00,PENDING_PAYMENT,LATAM
8,11,918,2015-01-01 03:30:00,PAYMENT_REVIEW,LATAM
9,12,1837,2015-01-01 03:51:00,CLOSED,LATAM


#### 5.1.1. Filtrado de registros con `WHERE`

A continuación, se aplica una condición de filtrado mediante `WHERE` para obtener únicamente los pedidos que presentan un estado determinado. En este caso, se seleccionan los pedidos cuyo estado es `COMPLETE`.


In [21]:
# Consultar los pedidos cuyo estado es COMPLETE
consulta = """
SELECT 
    order_id,
    customer_id,
    order_date_dateorders,
    order_status,
    market
FROM orders
WHERE order_status = 'COMPLETE'
LIMIT 10;
"""

resultado = pd.read_sql_query(consulta, conn)

resultado

,order_id,customer_id,order_date_dateorders,order_status,market
0,5,11318,2015-01-01 01:24:00,COMPLETE,LATAM
1,7,4530,2015-01-01 02:06:00,COMPLETE,LATAM
2,15,2568,2015-01-01 04:54:00,COMPLETE,LATAM
3,17,2667,2015-01-01 05:36:00,COMPLETE,LATAM
4,28,656,2015-01-01 09:27:00,COMPLETE,LATAM
5,35,4840,2015-01-01 11:54:00,COMPLETE,LATAM
6,45,2636,2015-01-01 15:24:00,COMPLETE,LATAM
7,56,10519,2015-01-01 19:16:00,COMPLETE,LATAM
8,63,1148,2015-01-01 21:43:00,COMPLETE,LATAM
9,65,5903,2015-01-01 22:25:00,COMPLETE,LATAM


#### 5.1.2. Ordenación de resultados con `ORDER BY`

Por último, se utiliza `ORDER BY` para ordenar los registros según una determinada variable. En este caso, se seleccionan los pedidos completados y se ordenan por fecha de pedido de forma descendente para identificar los más recientes.


In [22]:
# Consultar los pedidos COMPLETE más recientes
consulta = """
SELECT 
    order_id,
    customer_id,
    order_date_dateorders,
    order_status,
    market
FROM orders
WHERE order_status = 'COMPLETE'
ORDER BY order_date_dateorders DESC
LIMIT 10;
"""

resultado = pd.read_sql_query(consulta, conn)

resultado

,order_id,customer_id,order_date_dateorders,order_status,market
0,77202,20755,2018-01-31 22:56:00,COMPLETE,Pacific Asia
1,77201,20754,2018-01-31 22:35:00,COMPLETE,Pacific Asia
2,77197,20750,2018-01-31 21:11:00,COMPLETE,Pacific Asia
3,77196,20749,2018-01-31 20:50:00,COMPLETE,Pacific Asia
4,77190,20743,2018-01-31 18:44:00,COMPLETE,Pacific Asia
5,77189,20742,2018-01-31 18:23:00,COMPLETE,Pacific Asia
6,77185,20738,2018-01-31 16:59:00,COMPLETE,Pacific Asia
7,77183,20736,2018-01-31 16:17:00,COMPLETE,Pacific Asia
8,77181,20734,2018-01-31 15:35:00,COMPLETE,Pacific Asia
9,77176,20729,2018-01-31 13:50:00,COMPLETE,Pacific Asia


In [23]:
consulta_items = """
SELECT * 
FROM order_items 
LIMIT 10;
"""

df_items_prueba = pd.read_sql_query(consulta_items, conn)
df_items_prueba

,order_item_id,order_id,product_card_id,order_item_quantity,order_item_product_price,order_item_discount,order_item_discount_rate,order_item_total,order_item_profit_ratio,benefit_per_order,order_profit_per_order,sales,sales_per_customer
0,1,1,957,1,299.980011,60.00,0.20,239.979996,0.37,88.790001,88.790001,299.980011,239.979996
1,2,2,1073,1,199.990005,6.00,0.03,193.990005,0.47,91.180000,91.180000,199.990005,193.990005
2,3,2,502,5,50.000000,22.50,0.09,227.500000,0.30,68.250000,68.250000,250.000000,227.500000
3,4,2,403,1,129.990005,22.10,0.17,107.889999,0.34,36.470001,36.470001,129.990005,107.889999
4,5,4,897,2,24.990000,9.00,0.18,40.980000,0.10,4.100000,4.100000,49.980000,40.980000
5,6,4,365,5,59.990002,3.00,0.01,296.950012,0.09,26.129999,26.129999,299.950012,296.950012
6,7,4,502,3,50.000000,27.00,0.18,123.000000,0.49,60.270000,60.270000,150.000000,123.000000
7,8,4,1014,4,49.980000,39.98,0.20,159.940002,0.21,33.590000,33.590000,199.919998,159.940002
8,9,5,957,1,299.980011,54.00,0.18,245.979996,0.38,92.239998,92.239998,299.980011,245.979996
9,10,5,365,5,59.990002,0.00,0.00,299.950012,0.48,143.979996,143.979996,299.950012,299.950012


In [25]:
consulta_enero_2015 = """
SELECT 
    o.order_date_dateorders,
    COUNT(DISTINCT o.order_id) as total_pedidos,
    SUM(oi.sales) as suma_total_sales
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_date_dateorders LIKE '2015-01%'
GROUP BY o.order_date_dateorders;
"""

df_verificacion = pd.read_sql_query(consulta_enero_2015, conn)
df_verificacion

,order_date_dateorders,total_pedidos,suma_total_sales
0,2015-01-01 00:00:00,1,299.980011
1,2015-01-01 00:21:00,1,579.980011
2,2015-01-01 01:03:00,1,699.850010
3,2015-01-01 01:24:00,1,1129.860039
4,2015-01-01 02:06:00,1,579.920013
...,...,...,...
1782,2015-01-31 21:40:00,1,249.990005
1783,2015-01-31 22:01:00,1,639.960022
1784,2015-01-31 22:22:00,1,1099.800018
1785,2015-01-31 23:25:00,1,289.940006


In [26]:
# Suma total real de las ventas de Enero de 2015
consulta_total_enero = """
SELECT SUM(oi.sales) as total_mes_enero_2015
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_date_dateorders LIKE '2015-01%';
"""
print(pd.read_sql_query(consulta_total_enero, conn))

   total_mes_enero_2015
0          1.051590e+06
